Import required library like Langchain, FAIss, Gradio.

In [6]:
# Installing required packages
!pip3 install langchainhub
!pip3 install langchain-openai
!pip3 install langchain
!pip3 install beautifulsoup4
!pip3 install langchain-community
!pip3 install faiss-cpu
!pip3 install -U langchain-community tavily-python
!pip3 install gradio_client
!pip3 install gradio
!pip3 install langchain_community



  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached zstandard-0.25.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (3.3 kB)
  Using cached jiter-0.13.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.1 MB/s  0:00:00
Using cached jiter-0.13.0-cp312-cp312-macosx_11_0_arm64.whl (318 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 26.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 16.7 MB/s  0:00:00
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached zstandard-0.25.0-cp312-cp312-macosx_11_0_arm64.whl (640 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [langchain-openai][langchain-openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain]/6 [langgraph]
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [12]:
# Necessary Imports
# import kagglehub
import csv
import pandas as pd
import math
import numpy as np
import os
import getpass
from langchain_core.output_parsers import StrOutputParser
# from google.colab import drive

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain.chains import create_retrieval_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser

In [65]:
# dataset = pd.read_csv('DataSet/dataset.csv')
# symptom_description = pd.read_csv('DataSet/symptom_description.csv')
# symptom_precaution = pd.read_csv('DataSet/symptom_precaution.csv')
# symptom_severity = pd.read_csv('DataSet/symptom-severity.csv')

dataset = pd.read_csv('DataSet/dataset.csv')
symptom_description = pd.read_csv('DataSet/symptom_Description.csv')   # capital D
symptom_precaution = pd.read_csv('DataSet/symptom_precaution.csv')
symptom_severity = pd.read_csv('DataSet/Symptom-severity.csv')          # capital S

In [21]:
dataset.head(50)
#disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Severity

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 4920 entries, 0 to 4919
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Disease     4920 non-null   str  
 1   Symptom_1   4920 non-null   str  
 2   Symptom_2   4920 non-null   str  
 3   Symptom_3   4920 non-null   str  
 4   Symptom_4   4572 non-null   str  
 5   Symptom_5   3714 non-null   str  
 6   Symptom_6   2934 non-null   str  
 7   Symptom_7   2268 non-null   str  
 8   Symptom_8   1944 non-null   str  
 9   Symptom_9   1692 non-null   str  
 10  Symptom_10  1512 non-null   str  
 11  Symptom_11  1194 non-null   str  
 12  Symptom_12  744 non-null    str  
 13  Symptom_13  504 non-null    str  
 14  Symptom_14  306 non-null    str  
 15  Symptom_15  240 non-null    str  
 16  Symptom_16  192 non-null    str  
 17  Symptom_17  72 non-null     str  
dtypes: str(18)
memory usage: 692.0 KB


In [23]:
symptom_severity.head(20)

,Symptom,weight
0,itching,1
1,skin_rash,3
2,nodal_skin_eruptions,4
3,continuous_sneezing,4
4,shivering,5
5,chills,3
6,joint_pain,3
7,stomach_pain,5
8,acidity,3
9,ulcers_on_tongue,4


In [24]:
symptom_severity.info()

<class 'pandas.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Symptom  133 non-null    str  
 1   weight   133 non-null    int64
dtypes: int64(1), str(1)
memory usage: 2.2 KB


In [25]:
symptom_description.info()

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Disease      41 non-null     str  
 1   Description  41 non-null     str  
dtypes: str(2)
memory usage: 788.0 bytes


In [26]:
symptom_description.head()

,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [27]:
symptom_precaution.info()

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Disease       41 non-null     str  
 1   Precaution_1  41 non-null     str  
 2   Precaution_2  41 non-null     str  
 3   Precaution_3  40 non-null     str  
 4   Precaution_4  40 non-null     str  
dtypes: str(5)
memory usage: 1.7 KB


In [29]:
symptom_precaution.head(10)

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths
5,GERD,avoid fatty spicy food,avoid lying down after eating,maintain healthy weight,exercise
6,Chronic cholestasis,cold baths,anti itch medicine,consult doctor,eat healthy
7,hepatitis A,Consult nearest hospital,wash hands through,avoid fatty spicy food,medication
8,Osteoarthristis,acetaminophen,consult nearest hospital,follow up,salt baths
9,(vertigo) Paroymsal Positional Vertigo,lie down,avoid sudden change in body,avoid abrupt head movment,relax


In [32]:
dataset = dataset.rename(columns={'disease': 'Disease'})
symptom_description = symptom_description.rename(columns={'disease':'Disease'})
symptom_precaution = symptom_precaution.rename(columns={'disease':'Disease'})
symptom_severity = symptom_severity.rename(columns={'disease':'Disease'})

In [66]:
# Melt the symptom columns into a single column for easier merging
symptom_cols = [col for col in dataset.columns if col.startswith('Symptom')]
dataset_melted = dataset.melt(id_vars=['Disease'], value_vars=symptom_cols, value_name='Symptom').drop(columns='variable')
dataset_melted = dataset_melted.dropna(subset=['Symptom'])
dataset_melted['Symptom'] = dataset_melted['Symptom'].str.strip()

In [67]:
dataset_melted.head()

,Disease,Symptom
0,Fungal infection,itching
1,Fungal infection,skin_rash
2,Fungal infection,itching
3,Fungal infection,itching
4,Fungal infection,itching


In [68]:
# Merge with symptom severity
symptom_severity['Symptom'] = symptom_severity['Symptom'].str.strip()
dataset_with_severity = dataset_melted.merge(symptom_severity, on='Symptom', how='left')

In [69]:
dataset_with_severity.head()

,Disease,Symptom,weight
0,Fungal infection,itching,1.0
1,Fungal infection,skin_rash,3.0
2,Fungal infection,itching,1.0
3,Fungal infection,itching,1.0
4,Fungal infection,itching,1.0


In [70]:
# Merge with symptom description (disease-level)
dataset_with_desc = dataset_with_severity.merge(symptom_description, on='Disease', how='left')

In [71]:
dataset_with_desc.head()

,Disease,Symptom,weight,Description
0,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv..."
1,Fungal infection,skin_rash,3.0,"In humans, fungal infections occur when an inv..."
2,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv..."
3,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv..."
4,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv..."


In [72]:
# Merge with symptom precaution (disease-level)
dataset_with_precaution = dataset_with_desc.merge(symptom_precaution, on='Disease', how='left')

In [73]:
dataset_with_precaution.head()

,Disease,Symptom,weight,Description,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv...",bath twice,use detol or neem in bathing water,keep infected area dry,use clean cloths
1,Fungal infection,skin_rash,3.0,"In humans, fungal infections occur when an inv...",bath twice,use detol or neem in bathing water,keep infected area dry,use clean cloths
2,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv...",bath twice,use detol or neem in bathing water,keep infected area dry,use clean cloths
3,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv...",bath twice,use detol or neem in bathing water,keep infected area dry,use clean cloths
4,Fungal infection,itching,1.0,"In humans, fungal infections occur when an inv...",bath twice,use detol or neem in bathing water,keep infected area dry,use clean cloths


In [74]:
# Group by disease to create a RAG-friendly text document per disease
def create_rag_document(group):
    disease = group['Disease'].iloc[0]
    symptoms = group['Symptom'].tolist()
    severities = group['weight'].tolist()
    description = group['Description'].iloc[0] if 'Description' in group.columns else 'N/A'
    
    precaution_cols = [col for col in group.columns if col.lower().startswith('precaution')]
    precautions = group[precaution_cols].iloc[0].dropna().tolist()
    
    symptom_severity_info = ', '.join([f"{s} (severity: {w})" for s, w in zip(symptoms, severities)])
    precaution_info = ' | '.join(precautions)
    
    document = (
        # f"Disease: {disease}\n"
        f"Description: {description}\n"
        f"Symptoms with Severity: {symptom_severity_info}\n"
        f"Precautions: {precaution_info}"
    )
    return pd.Series({'Disease': disease, 'Document': document})

In [ ]:
# print(create_rag_document(dataset_with_precaution[dataset_with_precaution['Disease'] == 'Fungal infection']))

Disease                                      Fungal infection
Document    Description: In humans, fungal infections occu...
dtype: str


In [ ]:
# disease_grouped = dataset_with_precaution.groupby('Disease')

In [ ]:
# print(create_rag_document(disease_grouped.get_group('AIDS')))

Disease                                                  AIDS
Document    Description: Acquired immunodeficiency syndrom...
dtype: str


In [75]:
rag_dataset = dataset_with_precaution.groupby('Disease').apply(create_rag_document).reset_index(drop=True)

KeyError: 'Disease'

In [ ]:
print(f"Total RAG documents: {len(rag_dataset)}")
print("\nSample Document:\n")
print(rag_dataset['Document'].iloc[0])